In [1]:
import json
import math
from collections import defaultdict, Counter


# ==========================================
# 1. LOAD DATASET
# ==========================================

def load_data(filename):
    with open(filename, "r", encoding="utf-8") as file:
        data = json.load(file)

    return data


train_data = load_data("train.json")
dev_data = load_data("dev.json")

print("Training sentences:", len(train_data))
print("Development sentences:", len(dev_data))


# ==========================================
# 2. TRAIN HMM
# ==========================================

transition_counts = defaultdict(Counter)
emission_counts = defaultdict(Counter)
tag_counts = Counter()

START = "<START>"
END = "<END>"

for item in train_data:

    words = item["sentence"]
    tags = item["labels"]

    previous_tag = START

    for word, tag in zip(words, tags):

        # Count transition
        transition_counts[previous_tag][tag] += 1

        # Count word emission
        emission_counts[tag][word.lower()] += 1

        # Count tag
        tag_counts[tag] += 1

        previous_tag = tag

    # Transition from final tag to END
    transition_counts[previous_tag][END] += 1


# ==========================================
# 3. CONVERT COUNTS TO PROBABILITIES
# ==========================================

transition_prob = defaultdict(dict)
emission_prob = defaultdict(dict)

# Transition probabilities
for previous_tag in transition_counts:

    total = sum(transition_counts[previous_tag].values())

    for tag in transition_counts[previous_tag]:

        transition_prob[previous_tag][tag] = (
            transition_counts[previous_tag][tag] / total
        )


# Emission probabilities
for tag in emission_counts:

    total = sum(emission_counts[tag].values())

    for word in emission_counts[tag]:

        emission_prob[tag][word] = (
            emission_counts[tag][word] / total
        )


tags_list = list(tag_counts.keys())


# ==========================================
# 4. VITERBI ALGORITHM
# ==========================================

def viterbi(words):

    words = [word.lower() for word in words]

    viterbi_table = []
    backpointer = []

    # --------------------------------------
    # First word
    # --------------------------------------

    first_word = words[0]

    current_scores = {}
    current_backpointer = {}

    for tag in tags_list:

        transition = transition_prob[START].get(tag, 1e-10)

        emission = emission_prob[tag].get(first_word, 1e-10)

        score = math.log(transition) + math.log(emission)

        current_scores[tag] = score
        current_backpointer[tag] = None

    viterbi_table.append(current_scores)
    backpointer.append(current_backpointer)


    # --------------------------------------
    # Remaining words
    # --------------------------------------

    for word in words[1:]:

        previous_scores = current_scores

        current_scores = {}
        current_backpointer = {}

        for current_tag in tags_list:

            emission = emission_prob[current_tag].get(word, 1e-10)

            best_score = float("-inf")
            best_previous_tag = None

            for previous_tag in previous_scores:

                transition = transition_prob[previous_tag].get(
                    current_tag, 1e-10
                )

                score = (
                    previous_scores[previous_tag]
                    + math.log(transition)
                    + math.log(emission)
                )

                if score > best_score:

                    best_score = score
                    best_previous_tag = previous_tag

            current_scores[current_tag] = best_score
            current_backpointer[current_tag] = best_previous_tag

        viterbi_table.append(current_scores)
        backpointer.append(current_backpointer)


    # --------------------------------------
    # Find best final tag
    # --------------------------------------

    best_final_tag = max(
        current_scores,
        key=current_scores.get
    )

    best_tags = [best_final_tag]


    # --------------------------------------
    # Backtracking
    # --------------------------------------

    for i in range(len(words) - 1, 0, -1):

        previous_tag = backpointer[i][best_tags[-1]]

        best_tags.append(previous_tag)

    best_tags.reverse()

    return best_tags


# ==========================================
# 5. TEST WITH USER INPUT
# ==========================================

print("\nHMM POS TAGGER")
print("=" * 40)

sentence = input("Enter a sentence: ")

words = sentence.split()

predicted_tags = viterbi(words)


print("\nPOS Tags:")
print("-" * 40)

for word, tag in zip(words, predicted_tags):

    print(f"{word} -> {tag}")

Training sentences: 38218
Development sentences: 5527

HMM POS TAGGER


Enter a sentence:  The cat is sleeping



POS Tags:
----------------------------------------
The -> DT
cat -> NNP
is -> VBZ
sleeping -> VBG
